# Notebook 02 - Profile Operational Data

## Objective

Analyze the Bronze layer to understand the quality of the operational data before applying any transformations.

This notebook profiles each Bronze table to identify data quality issues such as missing values, duplicate records, inconsistent values, and record counts.

The results of this analysis will guide the transformations performed in the Silver layer.

## Pipeline Position
This is **Notebook 02** in the Medallion pipeline. It reads raw `bronze_*` Delta tables generated by Notebook 01, performs non-mutating data profiling to identify quality defects (nulls, duplicates, casing variations, margin anomalies), and establishes the data cleansing requirements implemented in Notebook 03 (Silver layer).

## Scope

- Record counts
- Missing values
- Duplicate records
- Distinct values
- Data quality observations

No data is modified in this notebook.


# Section 1 - Bronze Layer Record Counts

## Objective
Establish baseline record counts across all six ingested `bronze_*` Delta tables to confirm completeness against Notebook 01 raw file outputs.

In [1]:
tables = [
    "bronze_customers",
    "bronze_products",
    "bronze_orders",
    "bronze_order_lines",
    "bronze_campaign_performance",
    "bronze_sales_targets"
]

for table in tables:
    count = spark.table(table).count()
    print(f"{table}: {count} records")

StatementMeta(, 2a0edb5e-fe54-4f18-9348-6855f61ce5c5, 3, Finished, Available, Finished, False)

bronze_customers: 1002 records
bronze_products: 51 records
bronze_orders: 1000 records
bronze_order_lines: 2517 records
bronze_campaign_performance: 301 records
bronze_sales_targets: 48 records


# Section 2 - Customer Data Profiling

## Objective
Profile `bronze_customers` for duplicate `CustomerID` keys, missing email records, and casing variations across region names (e.g., `NORTH` vs `North`).

In [2]:
from pyspark.sql.functions import col, count

df = spark.table("bronze_customers")

print("Total Records:", df.count())

print("\nMissing Email")
df.filter(col("Email").isNull()).show()

print("\nDuplicate CustomerID")
df.groupBy("CustomerID") \
  .count() \
  .filter(col("count") > 1) \
  .show()

print("\nRegion Values")
df.groupBy("Region").count().show()

StatementMeta(, 2a0edb5e-fe54-4f18-9348-6855f61ce5c5, 4, Finished, Available, Finished, False)

Total Records: 1002

Missing Email
+----------+------------+------------+-----+----------+------+
|CustomerID|CustomerName|CustomerType|Email|  JoinDate|Region|
+----------+------------+------------+-----+----------+------+
|     10011| Customer 11|      Retail| NULL|2025-02-16|  West|
+----------+------------+------------+-----+----------+------+


Duplicate CustomerID
+----------+-----+
|CustomerID|count|
+----------+-----+
|     10051|    2|
|     10121|    2|
+----------+-----+


Region Values
+------+-----+
|Region|count|
+------+-----+
| South|  255|
| NORTH|    1|
|  East|  263|
|  West|  238|
| North|  245|
+------+-----+



# Section 3 - Product & Margin Profiling

## Objective
Inspect `bronze_products` for missing product categories, duplicate keys, and negative margin anomalies where `UnitPrice < StandardCost`.

In [3]:
df = spark.table("bronze_products")

print("Total Records:", df.count())

print("\nMissing Category")
df.filter(col("Category").isNull()).show()

print("\nDuplicate ProductID")
df.groupBy("ProductID") \
  .count() \
  .filter(col("count") > 1) \
  .show()

print("\nCategory Values")
df.groupBy("Category").count().show()

StatementMeta(, 2a0edb5e-fe54-4f18-9348-6855f61ce5c5, 5, Finished, Available, Finished, False)

Total Records: 51

Missing Category
+--------+---------+-----------+------------+---------+
|Category|ProductID|ProductName|StandardCost|UnitPrice|
+--------+---------+-----------+------------+---------+
|    NULL|        9|  Product 9|       22.36|   107.83|
+--------+---------+-----------+------------+---------+


Duplicate ProductID
+---------+-----+
|ProductID|count|
+---------+-----+
|       21|    2|
+---------+-----+


Category Values
+---------+-----+
| Category|count|
+---------+-----+
|  Kitchen|   14|
|   Office|   15|
|     NULL|    1|
| Cleaning|   14|
|Drinkware|    7|
+---------+-----+



In [3]:
from pyspark.sql.functions import col, when, count

products = spark.table("bronze_products")

total = products.count()

# 1. Unit price lower than standard cost
unit_price_below_cost = products.filter(
    col("UnitPrice") < col("StandardCost")
).count()

# 2. Unit price greater than standard cost
unit_price_above_cost = products.filter(
    col("UnitPrice") > col("StandardCost")
).count()

# 3. Same price and cost
same_price_cost = products.filter(
    col("UnitPrice") == col("StandardCost")
).count()

print("Total Products:", total)
print("UnitPrice < StandardCost:", unit_price_below_cost)
print("UnitPrice > StandardCost:", unit_price_above_cost)
print("UnitPrice = StandardCost:", same_price_cost)

print("\nPricing Relationship")
products.select(
    "ProductID",
    "ProductName",
    "StandardCost",
    "UnitPrice",
    when(col("UnitPrice") < col("StandardCost"), "Below Cost")
    .when(col("UnitPrice") > col("StandardCost"), "Above Cost")
    .otherwise("Equal")
    .alias("PricingStatus")
).show(100, truncate=False)

StatementMeta(, b6b56a0b-1fae-44d3-a0ff-21c16464be70, 5, Finished, Available, Finished, False)

Total Products: 51
UnitPrice < StandardCost: 28
UnitPrice > StandardCost: 23
UnitPrice = StandardCost: 0

Pricing Relationship
+---------+-----------+------------+---------+-------------+
|ProductID|ProductName|StandardCost|UnitPrice|PricingStatus|
+---------+-----------+------------+---------+-------------+
|1        |Product 1  |43.7        |32.58    |Below Cost   |
|2        |Product 2  |54.64       |83.05    |Above Cost   |
|3        |Product 3  |6.42        |36.64    |Below Cost   |
|4        |Product 4  |37.9        |66.04    |Above Cost   |
|5        |Product 5  |12.89       |33.55    |Above Cost   |
|6        |Product 6  |74.41       |69.79    |Below Cost   |
|7        |Product 7  |42.62       |52.97    |Above Cost   |
|8        |Product 8  |22.14       |22.04    |Below Cost   |
|9        |Product 9  |22.36       |107.83   |Below Cost   |
|10       |Product 10 |63.02       |32.33    |Below Cost   |
|11       |Product 11 |26.34       |110.6    |Below Cost   |
|12       |Product 

In [4]:
products.filter(
    col("UnitPrice") < col("StandardCost")
).select(
    "ProductID",
    "ProductName",
    "Category",
    "StandardCost",
    "UnitPrice"
).show(100, truncate=False)

StatementMeta(, b6b56a0b-1fae-44d3-a0ff-21c16464be70, 6, Finished, Available, Finished, False)

+---------+-----------+---------+------------+---------+
|ProductID|ProductName|Category |StandardCost|UnitPrice|
+---------+-----------+---------+------------+---------+
|1        |Product 1  |Cleaning |43.7        |32.58    |
|3        |Product 3  |Office   |6.42        |36.64    |
|6        |Product 6  |Drinkware|74.41       |69.79    |
|8        |Product 8  |Kitchen  |22.14       |22.04    |
|9        |Product 9  |NULL     |22.36       |107.83   |
|10       |Product 10 |Kitchen  |63.02       |32.33    |
|11       |Product 11 |Cleaning |26.34       |110.6    |
|12       |Product 12 |Cleaning |51.47       |28.62    |
|13       |Product 13 |Drinkware|52.2        |25.29    |
|14       |Product 14 |Kitchen  |62.78       |117.05   |
|16       |Product 16 |Drinkware|22.3        |102.89   |
|18       |Product 18 |Kitchen  |27.49       |107.74   |
|20       |Product 20 |Office   |67.05       |37.52    |
|23       |Product 23 |Kitchen  |69.41       |104.34   |
|25       |Product 25 |Kitchen 

# Section 4 - Orders & Line Items Profiling

## Objective
Verify channel distribution across sales order headers and audit `bronze_order_lines` for missing financial amounts or invalid quantity entries.

In [4]:
df = spark.table("bronze_orders")

print("Total Records:", df.count())

print("\nSales Channels")
df.groupBy("Channel").count().show()

StatementMeta(, 2a0edb5e-fe54-4f18-9348-6855f61ce5c5, 6, Finished, Available, Finished, False)

Total Records: 1000

Sales Channels
+---------+-----+
|  Channel|count|
+---------+-----+
|Wholesale|  351|
|   Online|  328|
|   Retail|  321|
+---------+-----+



**Order Lines Profile**

In [2]:
from pyspark.sql.functions import col

df = spark.table("bronze_order_lines")

print("Total Records:", df.count())

print("\nMissing UnitPrice")
df.filter(col("UnitPrice").isNull()).show()

print("\nMissing CostAtSale")
df.filter(col("CostAtSale").isNull()).show()

print("\nMissing LineAmount")
df.filter(col("LineAmount").isNull()).show()

print("\nQuantity Distribution")
df.groupBy("Quantity").count().show()

StatementMeta(, f5f755aa-ed18-49de-a48f-22b110c47bb7, 4, Finished, Available, Finished, False)

Total Records: 2497

Missing UnitPrice
+----------+--------+----------+-------+-----------+---------+--------+---------+
|CostAtSale|Discount|LineAmount|OrderID|OrderLineID|ProductID|Quantity|UnitPrice|
+----------+--------+----------+-------+-----------+---------+--------+---------+
+----------+--------+----------+-------+-----------+---------+--------+---------+


Missing CostAtSale
+----------+--------+----------+-------+-----------+---------+--------+---------+
|CostAtSale|Discount|LineAmount|OrderID|OrderLineID|ProductID|Quantity|UnitPrice|
+----------+--------+----------+-------+-----------+---------+--------+---------+
+----------+--------+----------+-------+-----------+---------+--------+---------+


Missing LineAmount
+----------+--------+----------+-------+-----------+---------+--------+---------+
|CostAtSale|Discount|LineAmount|OrderID|OrderLineID|ProductID|Quantity|UnitPrice|
+----------+--------+----------+-------+-----------+---------+--------+---------+
+----------+-----

# Section 5 - Marketing Campaign Profiling

## Objective
Profile `bronze_campaign_performance` to identify missing platform tags, casing variations in product categories, and duplicate campaign records.

In [2]:
from pyspark.sql.functions import col

df = spark.table("bronze_campaign_performance")

print("Total Records:", df.count())

print("\nMissing Platform")
df.filter(col("Platform").isNull()).show()

print("\nProduct Categories")
df.groupBy("ProductCategory").count().show()

print("\nDuplicate CampaignID")
df.groupBy("CampaignID") \
  .count() \
  .filter(col("count") > 1) \
  .show()

StatementMeta(, 86f4109d-e518-4696-ae10-a1f0c537ba03, 4, Finished, Available, Finished, False)

Total Records: 301

Missing Platform
+------------+----------+------------+------+-----------+-----------+--------+---------------+----------------+------+
|CampaignDate|CampaignID|CampaignName|Clicks|Conversions|Impressions|Platform|ProductCategory|RevenueGenerated| Spend|
+------------+----------+------------+------+-----------+-----------+--------+---------------+----------------+------+
|  2025-07-01|        11| Campaign 11|  4318|        304|      37335|    NULL|      Drinkware|         1257.63|394.78|
+------------+----------+------------+------+-----------+-----------+--------+---------------+----------------+------+


Product Categories
+---------------+-----+
|ProductCategory|count|
+---------------+-----+
|        Kitchen|   77|
|         Office|   92|
|      drinkware|    1|
|       Cleaning|   59|
|      Drinkware|   72|
+---------------+-----+


Duplicate CampaignID
+----------+-----+
|CampaignID|count|
+----------+-----+
|        31|    2|
+----------+-----+



# Section 6 - Sales Targets Profiling

## Objective
Audit monthly target planning records in `bronze_sales_targets` for null target values, inconsistent region casing, and `Unknown` category assignments.

In [3]:
from pyspark.sql.functions import col

df = spark.table("bronze_sales_targets")

print("Total Records:", df.count())

print("\nMissing Sales Targets")
df.filter(col("SalesTarget").isNull()).show()

print("\nRegion Values")
df.groupBy("Region").count().show()

print("\nProduct Categories")
df.groupBy("ProductCategory").count().show()

StatementMeta(, 86f4109d-e518-4696-ae10-a1f0c537ba03, 5, Finished, Available, Finished, True)

Total Records: 221

Missing Sales Targets
+-----+---------------+------+-----------+--------+----+
|Month|ProductCategory|Region|SalesTarget|TargetID|Year|
+-----+---------------+------+-----------+--------+----+
|    1|      Drinkware| North|       NULL|       6|2025|
+-----+---------------+------+-----------+--------+----+


Region Values
+------+-----+
|Region|count|
+------+-----+
| South|   56|
| NORTH|    1|
|  East|   56|
|  West|   55|
| North|   53|
+------+-----+


Product Categories
+---------------+-----+
|ProductCategory|count|
+---------------+-----+
|        Kitchen|   48|
|         Office|   47|
|        Unknown|   29|
|      drinkware|    1|
|       Cleaning|   48|
|      Drinkware|   48|
+---------------+-----+

